In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd 
import yaml
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go


In [36]:
with open("../config.yaml", "r") as file:
    config = yaml.safe_load(file)


In [4]:
df_cc = pd.read_csv(config["input_data"]["file1"])
df_cc.head()

,year,country,global_avg_temperature,temperature_anomaly,max_temperature,min_temperature,co2_concentration_ppm,annual_rainfall_mm,sea_level_rise_mm,sea_surface_temperature,heatwave_days,drought_index,flood_events_count,forest_cover_percent,deforestation_rate,fossil_fuel_consumption,renewable_energy_share,air_quality_index,predicted_temperature_2050,climate_risk_index
0,2018,Germany,14.03,1.16,37.02,4.31,387.85,814.11,2.14,14.38,50,0.85,11,64.33,1.26,28.14,42.30,148,3.30,23.69
1,2008,India,15.03,1.05,32.25,-0.44,407.24,735.61,4.19,15.33,37,1.01,11,42.69,0.62,86.40,53.65,50,2.63,70.10
2,1994,Pakistan,14.86,1.24,41.57,1.08,450.54,1982.92,6.46,15.90,27,4.59,1,65.47,2.08,34.95,47.20,107,1.33,34.74
3,2022,USA,15.29,1.19,32.17,1.68,415.42,1162.01,3.37,17.26,25,4.29,6,44.34,2.15,77.11,36.65,129,2.51,45.59
4,1987,Australia,13.75,1.32,40.99,10.38,403.42,1170.25,4.64,16.62,41,1.00,4,14.18,0.76,37.27,50.35,149,2.87,55.77


In [ ]:
#see all values of columns
pd.set_option('display.max_rows', None)
# Reset option to default if needed
#pd.reset_option('display.max_rows')

In [5]:
df_cc.info()


<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   year                        1200 non-null   int64  
 1   country                     1200 non-null   str    
 2   global_avg_temperature      1200 non-null   float64
 3   temperature_anomaly         1200 non-null   float64
 4   max_temperature             1200 non-null   float64
 5   min_temperature             1200 non-null   float64
 6   co2_concentration_ppm       1200 non-null   float64
 7   annual_rainfall_mm          1200 non-null   float64
 8   sea_level_rise_mm           1200 non-null   float64
 9   sea_surface_temperature     1200 non-null   float64
 10  heatwave_days               1200 non-null   int64  
 11  drought_index               1200 non-null   float64
 12  flood_events_count          1200 non-null   int64  
 13  forest_cover_percent        1200 non-null   

In [22]:
df_cc['country'].unique()

<StringArray>
[  'Germany',     'India',  'Pakistan',       'USA', 'Australia',    'France',
    'Canada',     'China',    'Brazil',        'UK']
Length: 10, dtype: str

In [23]:
def clean_cc(df_cc):
    # Drop specified columns from the DataFrame
    df_cc = df_cc.drop(['temperature_anomaly', 'max_temperature', 'min_temperature', 'drought_index'], axis=1)
    
    # Sort the DataFrame by 'country' and 'year'
    df_sorted = df_cc.sort_values(by=['country', 'year'], ascending=[True, True])
    
    # Filter for 'Australia'
    australia_data = df_sorted[df_sorted['country'] == 'Australia']
    
    # Filter out rows between 1980 and 2014, inclusive
    filtered_data = australia_data[~australia_data['year'].between(1980, 2014)]
    
    # Return the filtered DataFrame and intermediate variable
    return filtered_data, australia_data

# Apply the function and unpack the results
df_cc_cleaned, australia_data = clean_cc(df_cc)

# You can now use australia_data in further analysis
display(df_cc_cleaned)

,year,country,global_avg_temperature,co2_concentration_ppm,annual_rainfall_mm,sea_level_rise_mm,sea_surface_temperature,heatwave_days,flood_events_count,forest_cover_percent,deforestation_rate,fossil_fuel_consumption,renewable_energy_share,air_quality_index,predicted_temperature_2050,climate_risk_index
293,2015,Australia,15.24,447.31,841.58,2.48,14.83,6,7,14.61,0.74,87.10,17.32,191,2.15,94.27
652,2015,Australia,14.46,433.52,1259.45,2.93,18.01,54,3,16.11,3.32,66.20,24.58,169,2.11,86.63
926,2015,Australia,14.36,392.40,856.90,2.64,16.39,32,11,42.81,2.99,79.80,42.45,63,2.50,99.93
492,2016,Australia,13.42,410.92,1091.50,4.65,14.36,33,6,67.46,1.57,26.07,55.46,180,1.78,59.28
651,2016,Australia,14.95,476.52,896.56,4.28,16.38,52,6,23.67,2.30,50.85,23.21,183,2.01,18.95
938,2017,Australia,15.17,412.90,1022.20,2.48,16.27,43,2,38.77,1.40,74.98,58.68,84,1.97,26.50
148,2018,Australia,14.71,430.61,1355.15,5.00,18.00,47,1,40.08,2.85,69.35,43.42,170,2.46,89.01
620,2018,Australia,14.11,403.34,1577.69,2.63,15.43,46,10,57.33,2.61,23.77,38.24,168,1.48,0.57
625,2018,Australia,14.84,427.55,815.39,3.06,15.58,39,14,61.14,2.34,87.20,47.35,89,2.49,80.57
770,2018,Australia,14.38,434.25,1029.84,3.06,15.81,11,7,14.81,0.62,43.19,55.24,76,2.20,79.54


In [26]:
# Convert your DataFrame to a long format suitable for plotting
melted_data_all = australia_data.melt(
    id_vars='year',
    value_vars=[
        'global_avg_temperature',
        'sea_level_rise_mm',
        'deforestation_rate',
        'fossil_fuel_consumption',
        'sea_surface_temperature'
    ],
    var_name='Metric',
    value_name='Value'
)

# Create the stream graph using all data
fig = px.area(
    melted_data_all,
    x='year',
    y='Value',
    color='Metric',
    title='Climate Data for Australia - Stream Graph',
    labels={'Value': 'Value'},
    line_group='Metric'
)

# Customize the plot for better aesthetics
fig.update_traces(mode="lines+markers")  # Smooth lines and markers
fig.update_layout(yaxis_title='Value', xaxis_title='Year')

# Display the plot
fig.show()

In [11]:
df_cc_cleaned

,year,country,global_avg_temperature,co2_concentration_ppm,annual_rainfall_mm,sea_level_rise_mm,sea_surface_temperature,heatwave_days,flood_events_count,forest_cover_percent,deforestation_rate,fossil_fuel_consumption,renewable_energy_share,air_quality_index,predicted_temperature_2050,climate_risk_index
293,2015,Australia,15.24,447.31,841.58,2.48,14.83,6,7,14.61,0.74,87.10,17.32,191,2.15,94.27
652,2015,Australia,14.46,433.52,1259.45,2.93,18.01,54,3,16.11,3.32,66.20,24.58,169,2.11,86.63
926,2015,Australia,14.36,392.40,856.90,2.64,16.39,32,11,42.81,2.99,79.80,42.45,63,2.50,99.93
492,2016,Australia,13.42,410.92,1091.50,4.65,14.36,33,6,67.46,1.57,26.07,55.46,180,1.78,59.28
651,2016,Australia,14.95,476.52,896.56,4.28,16.38,52,6,23.67,2.30,50.85,23.21,183,2.01,18.95
938,2017,Australia,15.17,412.90,1022.20,2.48,16.27,43,2,38.77,1.40,74.98,58.68,84,1.97,26.50
148,2018,Australia,14.71,430.61,1355.15,5.00,18.00,47,1,40.08,2.85,69.35,43.42,170,2.46,89.01
620,2018,Australia,14.11,403.34,1577.69,2.63,15.43,46,10,57.33,2.61,23.77,38.24,168,1.48,0.57
625,2018,Australia,14.84,427.55,815.39,3.06,15.58,39,14,61.14,2.34,87.20,47.35,89,2.49,80.57
770,2018,Australia,14.38,434.25,1029.84,3.06,15.81,11,7,14.81,0.62,43.19,55.24,76,2.20,79.54


In [27]:
# Convert your DataFrame to a long format suitable for plotting
melted_data_all = australia_data.melt(
    id_vars='year',
    value_vars=[
        'fossil_fuel_consumption',
        'sea_surface_temperature',
        'sea_level_rise_mm',
        'deforestation_rate',
         'global_avg_temperature'
    ],
    var_name='Metric',
    value_name='Value'
)

# Define a specific color for each metric
color_map = {
    'fossil_fuel_consumption': 'grey',
    'global_avg_temperature': 'red',
    'sea_surface_temperature': 'orange',
    'sea_level_rise_mm': 'blue',
    'deforestation_rate': 'green'
}

# Create the stream graph using all data
fig = px.area(
    melted_data_all,
    x='year',
    y='Value',
    color='Metric',
    title='Climate Data for Australia - Stream Graph',
    labels={'Value': 'Value'},
    line_group='Metric',
    color_discrete_map=color_map  # Assign specific colors
)

# Customize the plot for better aesthetics
fig.update_traces(mode="lines+markers")  # Smooth lines and markers
fig.update_layout(yaxis_title='Value', xaxis_title='Year')

# Display the plot
fig.show()

In [28]:

# Convert your DataFrame to a long format suitable for plotting
melted_data_all = australia_data.melt(
    id_vars='year',
    value_vars=[
        'fossil_fuel_consumption',
        'sea_surface_temperature',
        'sea_level_rise_mm',
        'deforestation_rate',
        'global_avg_temperature'
    ],
    var_name='Metric',
    value_name='Value'
)

# Define a specific color for each metric
color_map = {
    'fossil_fuel_consumption': 'grey',
    'global_avg_temperature': 'red',
    'sea_surface_temperature': 'orange',
    'sea_level_rise_mm': 'blue',
    'deforestation_rate': 'green'
}

# Prepare traces for each metric
traces = []
for metric, color in color_map.items():
    filtered_data = melted_data_all[melted_data_all['Metric'] == metric]
    trace = go.Scatter(
        x=filtered_data['year'],
        y=filtered_data['Value'],
        mode='lines+markers',
        name=metric,
        line=dict(color=color),
        fill='tozeroy',  # Fill from the line to the x-axis
        fillcolor=color,  # Match fill color to line color
        opacity=0.5  # Adjust opacity to allow visibility of overlaps
    )
    traces.append(trace)

# Define layout
layout = go.Layout(
    title='Climate Data for Australia - Non-Stacked Plot',
    xaxis=dict(title='Year'),
    yaxis=dict(title='Value'),
    hovermode='x unified'  # Improve hover info display
)

# Create the figure
fig = go.Figure(data=traces, layout=layout)

# Display the plot
fig.show()

In [29]:
# Data preparation remains the same
melted_data_all = australia_data.melt(
    id_vars='year',
    value_vars=[
        'fossil_fuel_consumption',
        'sea_surface_temperature',
        'sea_level_rise_mm',
        'deforestation_rate',
        'global_avg_temperature'
    ],
    var_name='Metric',
    value_name='Value'
)


# Prepare traces while modifying opacity and overlap to simulate flow
traces = []
for i, (metric, color) in enumerate(color_map.items()):
    filtered_data = melted_data_all[melted_data_all['Metric'] == metric]
    trace = go.Scatter(
        x=filtered_data['year'],
        y=filtered_data['Value'] + i * 5,  # Offset for more visual separation
        mode='lines',
        name=metric,
        line=dict(color=color, shape='spline'),  # Spline for smoothness
        fill='tonexty',  # Smooth visual perception
        fillcolor=color,
        opacity=0.6  # Translucency for flow effect
    )
    traces.append(trace)

# Define layout
layout = go.Layout(
    title='Approximating a Flowy Stream Graph',
    xaxis=dict(title='Year'),
    yaxis=dict(title='Value'),
    hovermode='closest'
)

# Create the figure
fig = go.Figure(data=traces, layout=layout)

# Display the plot
fig.show()

In [30]:
# Data preparation remains the same
melted_data_all = australia_data.melt(
    id_vars='year',
    value_vars=[
        'fossil_fuel_consumption',
        'sea_surface_temperature',
        'sea_level_rise_mm',
        'deforestation_rate',
        'global_avg_temperature'
    ],
    var_name='Metric',
    value_name='Value'
)

# Prepare traces without specific color mapping
traces = []
for metric in melted_data_all['Metric'].unique():
    filtered_data = melted_data_all[melted_data_all['Metric'] == metric]
    trace = go.Scatter(
        x=filtered_data['year'],
        y=filtered_data['Value'],
        mode='lines',
        name=metric,
        line=dict(shape='spline'),  # Spline for smooth appearance
        fill='tonexty',  # Visual depth effect with overlapping
        opacity=0.6  # Translucency for flow effect
    )
    traces.append(trace)

# Define layout
layout = go.Layout(
    title='Climate Data for Australia - Non-Stacked Stream-like Plot',
    xaxis=dict(title='Year'),
    yaxis=dict(title='Value'),
    hovermode='closest'
)

# Create the figure
fig = go.Figure(data=traces, layout=layout)

# Display the plot
fig.show()

In [31]:
# Data preparation remains the same
melted_data_all = australia_data.melt(
    id_vars='year',
    value_vars=[
        'sea_surface_temperature',
        'global_avg_temperature'
    ],
    var_name='Metric',
    value_name='Value'
)

# Prepare traces without specific color mapping
traces = []
for metric in melted_data_all['Metric'].unique():
    filtered_data = melted_data_all[melted_data_all['Metric'] == metric]
    trace = go.Scatter(
        x=filtered_data['year'],
        y=filtered_data['Value'],
        mode='lines',
        name=metric,
        line=dict(shape='spline'),  # Spline for smooth appearance
        fill='tonexty',  # Visual depth effect with overlapping
        opacity=0.6  # Translucency for flow effect
    )
    traces.append(trace)

# Define layout
layout = go.Layout(
    title='Climate Data for Australia - Non-Stacked Stream-like Plot',
    xaxis=dict(title='Year'),
    yaxis=dict(title='Value'),
    hovermode='closest'
)

# Create the figure
fig = go.Figure(data=traces, layout=layout)

# Display the plot
fig.show()

In [33]:
# Assuming 'color_map' is defined like this
color_map = {
    'sea_surface_temperature': 'blue',
    'global_avg_temperature': 'red'
}

# Convert your DataFrame to a long format suitable for plotting
melted_data_all = australia_data.melt(
    id_vars='year',
    value_vars=[
        'global_avg_temperature',
        'sea_surface_temperature'
    ],
    var_name='Metric',
    value_name='Value'
)

# Prepare traces for each metric, defining order for visibility
traces = []
for metric in ['sea_surface_temperature', 'global_avg_temperature']:  # Define specific order
    filtered_data = melted_data_all[melted_data_all['Metric'] == metric]
    trace = go.Scatter(
        x=filtered_data['year'],
        y=filtered_data['Value'],
        mode='lines+markers',
        name=metric,
        line=dict(color=color_map[metric]),
        fill='tozeroy',  # Fill from the line to the x-axis
        fillcolor=color_map[metric],
        opacity=0.5  # Adjust opacity to allow visibility of overlaps
    )
    traces.append(trace)

# Define layout
layout = go.Layout(
    title='Climate Data for Australia - Non-Stacked Plot',
    xaxis=dict(title='Year'),
    yaxis=dict(title='Value'),
    hovermode='x unified'  # Improve hover info display
)

# Create the figure
fig = go.Figure(data=traces, layout=layout)

# Display the plot
fig.show()

In [34]:
# Convert your DataFrame to a long format suitable for plotting
melted_data_all = australia_data.melt(
    id_vars='year',
    value_vars=[
       'sea_surface_temperature',
        'sea_level_rise_mm',
        'deforestation_rate',
        'global_avg_temperature'
    ],
    var_name='Metric',
    value_name='Value'
)

# Create a line graph using Plotly Express
fig = px.line(
    melted_data_all,
    x='year',
    y='Value',
    color='Metric',
    title='Climate Data for Australia - Line Graph',
    labels={'Value': 'Value'}
)

# Show the plot
fig.show()

In [35]:
# Convert your DataFrame to a long format suitable for plotting
melted_data_all = australia_data.melt(
    id_vars='year',
    value_vars=[
       'sea_surface_temperature',
        'deforestation_rate'
     
    ],
    var_name='Metric',
    value_name='Value'
)

# Create a line graph using Plotly Express
fig = px.line(
    melted_data_all,
    x='year',
    y='Value',
    color='Metric',
    title='Climate Data for Australia - Line Graph',
    labels={'Value': 'Value'}
)

# Show the plot
fig.show()

In [ ]:
df.to_csv(config[])